In [1]:
# Célula 1 — Imports
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [2]:
# Célula 2 — Carregar dados
pasta = Path(r"C:\Users\franc\OneDrive\Francisco\Profissional\MBA_Data_Science_ e_Analytics\00_TCC\06_Dados_base\GEO\2024_02_basegeo")

lf = gpd.read_file(pasta / "lf.shp")
bf = gpd.read_file(pasta / "bf.shp")
gl = gpd.read_file(pasta / "gl.shp")

lf["CAMADA"] = "LF"
bf["CAMADA"] = "BF"
gl["CAMADA"] = "GL"

lf_val = lf[lf.geometry.notna() & lf.geometry.is_valid].copy()
bf_val = bf[bf.geometry.notna() & bf.geometry.is_valid].copy()
gl_val = gl[gl.geometry.notna() & gl.geometry.is_valid].copy()

In [3]:
# Célula 3 — Calcular sobreposição real para TODOS os pares entre camadas
LIMIAR_BORDA = 5.0   # m² — abaixo disso é imprecisão de digitalização
LIMIAR_REAL  = 50.0  # m² — acima disso é sobreposição significativa

def analisar_sobreposicoes(gdf_a, nome_a, gdf_b, nome_b):
    print(f"\n{'='*70}")
    print(f"  {nome_a} × {nome_b}")
    print(f"{'='*70}")

    joined = gpd.sjoin(gdf_a, gdf_b, how="inner", predicate="overlaps")
    total_pares = len(joined)
    print(f"  Pares detectados pelo sjoin (overlaps): {total_pares:,}")

    if total_pares == 0:
        return

    resultados = []
    for idx, row in joined.iterrows():
        geom_a = gdf_a.loc[idx, "geometry"]
        idx_b  = row["index_right"]
        geom_b = gdf_b.loc[idx_b, "geometry"]

        if not geom_a.intersects(geom_b):
            continue

        intersec   = geom_a.intersection(geom_b)
        area_over  = intersec.area
        nb_a       = gdf_a.loc[idx, "NUMBLOCO"]
        nb_b       = gdf_b.loc[idx_b, "NUMBLOCO"]
        area_a     = geom_a.area
        area_b     = geom_b.area
        pct_a      = area_over / area_a * 100 if area_a > 0 else 0
        pct_b      = area_over / area_b * 100 if area_b > 0 else 0

        if area_over <= LIMIAR_BORDA:
            tipo = "BORDA"
        elif pct_b >= 90:
            tipo = "CONTIDO (B em A)"
        elif pct_a >= 90:
            tipo = "CONTIDO (A em B)"
        elif area_over >= LIMIAR_REAL:
            tipo = "SOBREPOSIÇÃO REAL"
        else:
            tipo = "PARCIAL"

        resultados.append({
            f"NUMBLOCO_{nome_a}": nb_a,
            f"AREA_{nome_a}":     round(area_a, 1),
            f"NUMBLOCO_{nome_b}": nb_b,
            f"AREA_{nome_b}":     round(area_b, 1),
            "AREA_OVERLAP":       round(area_over, 2),
            f"% {nome_a}":        round(pct_a, 1),
            f"% {nome_b}":        round(pct_b, 1),
            "TIPO":               tipo,
        })

    df = pd.DataFrame(resultados)
    resumo = df["TIPO"].value_counts()
    print(f"\n  Resumo por tipo:")
    for tipo, qtd in resumo.items():
        print(f"    {tipo:<25} {qtd:>6,} ocorrências")

    for tipo in ["CONTIDO (B em A)", "CONTIDO (A em B)", "SOBREPOSIÇÃO REAL"]:
        sub = df[df["TIPO"] == tipo]
        if len(sub) > 0:
            print(f"\n  Amostra — {tipo} (até 10 registros):")
            print(sub.head(10).to_string(index=False))

    return df

df_lf_bf = analisar_sobreposicoes(lf_val, "LF", bf_val, "BF")
df_lf_gl = analisar_sobreposicoes(lf_val, "LF", gl_val, "GL")
df_bf_gl = analisar_sobreposicoes(bf_val, "BF", gl_val, "GL")


  LF × BF


  Pares detectados pelo sjoin (overlaps): 23,798



  Resumo por tipo:
    BORDA                     12,596 ocorrências
    PARCIAL                    9,399 ocorrências
    SOBREPOSIÇÃO REAL          1,764 ocorrências
    CONTIDO (A em B)              20 ocorrências
    CONTIDO (B em A)              19 ocorrências

  Amostra — CONTIDO (B em A) (até 10 registros):
 NUMBLOCO_LF  AREA_LF  NUMBLOCO_BF  AREA_BF  AREA_OVERLAP  % LF  % BF             TIPO
000000000000    989.6 005006210005    973.9        972.70  98.3  99.9 CONTIDO (B em A)
000000000000   3768.7 007188830000    294.4        294.35   7.8 100.0 CONTIDO (B em A)
000000000000    305.0 006000570007    305.7        304.48  99.8  99.6 CONTIDO (B em A)
000000000000     37.3 000000000000     30.4         30.21  80.9  99.3 CONTIDO (B em A)
007002120001   9251.3 007002120004    462.0        461.85   5.0 100.0 CONTIDO (B em A)
000000000000    961.2 000000000000    327.5        327.54  34.1 100.0 CONTIDO (B em A)
002622160000  22888.5 000000000000   1190.9       1119.07   4.9  94.0 CONTID

  Pares detectados pelo sjoin (overlaps): 3,634



  Resumo por tipo:
    CONTIDO (A em B)           1,366 ocorrências
    SOBREPOSIÇÃO REAL            934 ocorrências
    PARCIAL                      791 ocorrências
    BORDA                        505 ocorrências
    CONTIDO (B em A)              38 ocorrências

  Amostra — CONTIDO (B em A) (até 10 registros):
 NUMBLOCO_LF  AREA_LF  NUMBLOCO_GL  AREA_GL  AREA_OVERLAP  % LF  % GL             TIPO
002002250000 768857.8 007087080000  68470.4      65428.64   8.5  95.6 CONTIDO (B em A)
000000000000 360593.8 007175850000   3844.6       3765.96   1.0  98.0 CONTIDO (B em A)
000000000000 360593.8 000000000000 362195.7     360593.68 100.0  99.6 CONTIDO (B em A)
002373880000   1721.9 002191180002   1658.2       1584.06  92.0  95.5 CONTIDO (B em A)
001507920000  15298.2 001507920000  15267.4      14545.90  95.1  95.3 CONTIDO (B em A)
000000000000   3545.4 002557550000   1756.5       1719.91  48.5  97.9 CONTIDO (B em A)
001825960000  10982.2 000000000000   1730.7       1641.26  14.9  94.8 CONTID

  Pares detectados pelo sjoin (overlaps): 3,709



  Resumo por tipo:
    CONTIDO (A em B)           1,565 ocorrências
    SOBREPOSIÇÃO REAL            811 ocorrências
    PARCIAL                      676 ocorrências
    BORDA                        647 ocorrências
    CONTIDO (B em A)              10 ocorrências

  Amostra — CONTIDO (B em A) (até 10 registros):
 NUMBLOCO_BF  AREA_BF  NUMBLOCO_GL  AREA_GL  AREA_OVERLAP  % BF  % GL             TIPO
000012930000   6633.0 007076710000     89.3         88.69   1.3  99.3 CONTIDO (B em A)
001599330000    561.1 001599320000    147.4        142.91  25.5  96.9 CONTIDO (B em A)
000000000000   3870.8 000000000000   3829.3       3826.28  98.8  99.9 CONTIDO (B em A)
001090640000   1233.4 001925160000    460.1        452.26  36.7  98.3 CONTIDO (B em A)
001766590000    442.0 000000000000    269.2        267.88  60.6  99.5 CONTIDO (B em A)
002101350000    596.0 001434160000    290.7        286.44  48.1  98.5 CONTIDO (B em A)
001638700000    629.9 001193290000    270.3        259.38  41.2  95.9 CONTID